# 📐 Topic 03: Padding & Strides from Scratch and Keras
### 🧮 Valid vs Same Padding, Stride Math, Custom NumPy Logic & Keras Verification

---

## 📌 1. Padding Kyun Zaroori Hai? (The Two Big Problems)

Jab hum bina padding ke raw image par convolution chalate hain, toh do major samasyaayein (problems) aati hain:

### 1️⃣ Problem 1: Shrinking Feature Maps (Image Choti Hoti Jaati Hai)
- Agar input image ka size $W \times W$ hai aur filter $F \times F$ hai, toh convolution ke baad output size hota hai:
  $$\text{Output Size} = W - F + 1$$
- Maan lijiye hamari image $28 \times 28$ hai aur hum $5 \times 5$ filter lagate hain:
  - Layer 1 ke baad: $28 - 5 + 1 = 24 \times 24$
  - Layer 2 ke baad: $24 - 5 + 1 = 20 \times 20$
  - Layer 3 ke baad: $20 - 5 + 1 = 16 \times 16$
  - Kuch hi layers mein image shrink hokar $1 \times 1$ ban jayegi! Hum deeper networks (jaise 50 ya 100 layers) bana hi nahi sakte.

### 2️⃣ Problem 2: Corner Pixels ki Under-Representation (Info Loss)
- Corner (kona) aur border wale pixels par filter sirf **ek baar** chalta hai.
- Jabki center (beech) wale pixels par filter **baar-baar ($F \times F$ times)** overlap hota hai.
- Iska matlab border par maujood important features (jaise object ki boundaries ya edges) ignore ho jaate hain.

---

## 🛡️ Padding ka Solution: Valid vs Same Padding

1. **Valid Padding ($P = 0$)**:
   - Iska matlab hota hai **No Padding** (koi extra pixel add nahi karte).
   - Output size chota hota hai: $O = W - F + 1$.
2. **Same Padding**:
   - Iska matlab hum image ke chaaron taraf itne $0$s (Zero-padding) add karte hain taaki **Output Size == Input Size** ho!
   - Agar stride $S = 1$ hai, toh kitna padding $P$ add karna padega?
     $$P = \frac{F - 1}{2}$$
   - *Example*: Agar filter $3 \times 3$ hai ($F=3$), toh $P = (3-1)/2 = 1$ pixel padding dono taraf chahiye.
   - *Example*: Agar filter $5 \times 5$ hai ($F=5$), toh $P = (5-1)/2 = 2$ pixels padding chahiye.


In [ ]:
# ============================================================
# STEP 1: Libraries Import
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow Version: {tf.__version__}")


In [ ]:
# ============================================================
# STEP 2: Custom Padding & Strided Convolution from Scratch
# ============================================================

def pad_image_scratch(image, pad_h, pad_w, pad_value=0):
    """Image ke chaaron taraf custom padding add karta hai"""
    H, W = image.shape
    padded = np.full((H + 2 * pad_h, W + 2 * pad_w), fill_value=pad_value, dtype=image.dtype)
    padded[pad_h:pad_h + H, pad_w:pad_w + W] = image
    return padded

def conv2d_custom(image, kernel, padding=0, stride=1):
    """
    Custom Conv2D with explicit padding and stride
    Output formula: floor((W - F + 2P)/S) + 1
    """
    if padding > 0:
        img_padded = pad_image_scratch(image, padding, padding, pad_value=0)
    else:
        img_padded = image
        
    H, W = img_padded.shape
    kH, kW = kernel.shape
    
    out_H = (H - kH) // stride + 1
    out_W = (W - kW) // stride + 1
    
    output = np.zeros((out_H, out_W), dtype=np.float32)
    
    for i in range(out_H):
        for j in range(out_W):
            start_i = i * stride
            start_j = j * stride
            patch = img_padded[start_i:start_i + kH, start_j:start_j + kW]
            output[i, j] = np.sum(patch * kernel)
            
    return output

# Testing on a simple 5x5 grid
grid = np.arange(1, 26).reshape((5, 5)).astype(np.float32)
kernel = np.ones((3, 3), dtype=np.float32)

print("Original 5x5 Grid:\n", grid)
print("\n1. Valid Padding (P=0, S=1):\n", conv2d_custom(grid, kernel, padding=0, stride=1).shape)
print("2. Same Padding (P=1, S=1):\n", conv2d_custom(grid, kernel, padding=1, stride=1).shape)
print("3. Stride=2 with Same Padding (P=1, S=2):\n", conv2d_custom(grid, kernel, padding=1, stride=2).shape)


### 💡 Shape Calculations Analysis:
1. **Valid ($P=0, S=1$)**: $(5 - 3 + 0)/1 + 1 = 3 \times 3$ $\rightarrow$ Output shrunk from $5 \times 5$ to $3 \times 3$.
2. **Same ($P=1, S=1$)**: $(5 - 3 + 2)/1 + 1 = 5 \times 5$ $\rightarrow$ Output remains exact $5 \times 5$!
3. **Stride=2 ($P=1, S=2$)**: $\lfloor (5 - 3 + 2)/2 \rfloor + 1 = \lfloor 4/2 \rfloor + 1 = 3 \times 3$ $\rightarrow$ Spatial dimension downsampled!

## 🖼️ 3. Visual Demonstration on Real Image

Aaiye ek digit image par padding aur stride ka visual asar dekhte hain:

In [ ]:
# ============================================================
# STEP 3: Visual Demonstration on MNIST Image
# ============================================================

mnist = keras.datasets.mnist
(X_train, _), _ = mnist.load_data()
sample_digit = X_train[0].astype(np.float32) / 255.0  # (28, 28)

# 1. Padded Image Visual (P=4 taaki border clearly visible ho)
padded_digit = pad_image_scratch(sample_digit, pad_h=4, pad_w=4, pad_value=0.5)

# 2. Edge filter
edge_filter = np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]], dtype=np.float32)

# 3. Convolution with Stride 1 vs Stride 2
out_stride1 = conv2d_custom(sample_digit, edge_filter, padding=1, stride=1)
out_stride2 = conv2d_custom(sample_digit, edge_filter, padding=1, stride=2)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(sample_digit, cmap='gray')
axes[0].set_title(f"Original Image\nShape: {sample_digit.shape}", fontweight='bold')
axes[0].axis('on')

axes[1].imshow(padded_digit, cmap='gray')
axes[1].set_title(f"Padded Border (Gray border)\nShape: {padded_digit.shape}", fontweight='bold')
axes[1].axis('on')

axes[2].imshow(out_stride1, cmap='gray')
axes[2].set_title(f"Same Padding (Stride=1)\nShape: {out_stride1.shape}", fontweight='bold')
axes[2].axis('on')

axes[3].imshow(out_stride2, cmap='gray')
axes[3].set_title(f"Same Padding (Stride=2)\nShape: {out_stride2.shape}", fontweight='bold')
axes[3].axis('on')

plt.suptitle("Padding aur Strides ka Visual Impact", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 🚀 4. Keras Implementation (`Conv2D` & `ZeroPadding2D`)

Keras mein hum do tareeqon se padding control kar sakte hain:
1. `layers.ZeroPadding2D(padding=((top, bottom), (left, right)))`
2. `layers.Conv2D(..., padding='valid' ya 'same', strides=(s, s))`

Aaiye Keras ke output shapes verify karte hain:

In [ ]:
# ============================================================
# STEP 4: Keras Conv2D Padding & Stride Verification
# ============================================================

input_shape = (28, 28, 1)
dummy_x = tf.zeros((1, 28, 28, 1))

# Case 1: Valid Padding, Stride 1
conv_valid = layers.Conv2D(16, kernel_size=(3, 3), padding='valid', strides=1)
out1 = conv_valid(dummy_x)
print(f"1. Valid Padding (Stride 1): Input {input_shape} -> Output {out1.shape[1:]} (Formula: 28-3+1 = 26)")

# Case 2: Same Padding, Stride 1
conv_same = layers.Conv2D(16, kernel_size=(3, 3), padding='same', strides=1)
out2 = conv_same(dummy_x)
print(f"2. Same Padding  (Stride 1): Input {input_shape} -> Output {out2.shape[1:]} (Shape exact 28x28 preserved!)")

# Case 3: Same Padding, Stride 2
conv_stride2 = layers.Conv2D(16, kernel_size=(3, 3), padding='same', strides=2)
out3 = conv_stride2(dummy_x)
print(f"3. Same Padding  (Stride 2): Input {input_shape} -> Output {out3.shape[1:]} (Half downsampled to 14x14!)")

# Case 4: Explicit ZeroPadding2D Layer (Jaise LeNet-5 mein 28x28 ko 32x32 kiya tha)
pad_layer = layers.ZeroPadding2D(padding=((2, 2), (2, 2)))
out4 = pad_layer(dummy_x)
print(f"4. ZeroPadding2D Layer:      Input {input_shape} -> Output {out4.shape[1:]} (LeNet-5 32x32 ready!)")


## 💡 Universal Formula Cheat-Sheet (Interview Ready):

$$\text{Output Dimension} = \left\lfloor \frac{W - F + 2P}{S} \right\rfloor + 1$$

| Case | Padding Formula | When to Use? |
| :--- | :--- | :--- |
| **Valid** | $P = 0$ | Jab feature maps ko shrink karna acceptable ya desired ho |
| **Same** | $P = (F - 1) / 2$ | Jab dimensions preserve rakhna ho (deeper networks jaise VGG, ResNet) |
| **Stride > 1** | Downsamples by factor of $S$ | Pooling ki jagah computational cost aur size reduce karne ke liye |

---

## 📝 Summary & Key Takeaways in Hinglish

1. **Information Loss Prevention**: Padding border pixels ko equal importance deti hai aur unka information preserve karti hai.
2. **Deep Networks**: Same padding ke bina hum deep networks nahi bana sakte kyunki feature maps bahut jaldi shrink ho jaate hain.
3. **Stride as Downsampler**: Stride 2 image ko half size ($50\%$) kar deta hai, jisse computation fast hoti hai.
